<a href="https://colab.research.google.com/github/kashish2610/Pytorch/blob/main/PASCAL_VOC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name     :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
if not torch.cuda.is_available():

    print("No GPU found!")

else:
    print("GPU is ready")

GPU available: True
GPU name     : Tesla T4
GPU is ready


In [ ]:
from google.colab import files
import os

os.chdir('/content')
uploaded = files.upload()

print()
for name in uploaded:
    with open(f'/content/{name}', 'wb') as f:
        f.write(uploaded[name])
    print(f"{name}")

print()
print("All files saved to /content/")

Saving dataset.py to dataset.py
Saving EfficientSeg_v3_Colab.ipynb to EfficientSeg_v3_Colab.ipynb
Saving inference.py to inference.py
Saving losses.py to losses.py
Saving model.py to model.py
Saving README.md to README.md
Saving requirements.txt to requirements.txt
Saving train.py to train.py
Saving validate.py to validate.py

dataset.py
EfficientSeg_v3_Colab.ipynb
inference.py
losses.py
model.py
README.md
requirements.txt
train.py
validate.py

All files saved to /content/


In [8]:
!pip install -r requirements.txt

In [9]:
import tarfile
import urllib.request
import os

VOC_URL = 'http://host.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar'
VOC_DIR = '/content/VOCdevkit/VOC2012'

if not os.path.exists(VOC_DIR):
    print('Downloading PASCAL VOC 2012...')
    urllib.request.urlretrieve(VOC_URL, '/content/voc2012.tar')
    print('Extracting...')
    with tarfile.open('/content/voc2012.tar') as tar:
        tar.extractall('/content/')
    os.remove('/content/voc2012.tar')
    print('Done!')
else:
    print('VOC 2012 already exists.')

print(f"Images: {len(os.listdir(os.path.join(VOC_DIR, 'JPEGImages')))}")
print(f"Seg masks: {len(os.listdir(os.path.join(VOC_DIR, 'SegmentationClass')))}")

VOC 2012 already exists.
Images: 17125
Seg masks: 2913


In [ ]:
import torch
import torch.nn.functional as F
import losses, importlib, sys

# Force reload from disk
if 'losses' in sys.modules:
    del sys.modules['losses']
importlib.invalidate_caches()

# Rewrite the file correctly
with open('/content/losses.py', 'r') as f:
    src = f.read()

# Nuclear option — replace the entire _edges method by line
lines = src.split('\n')
new_lines = []
skip = 0
for i, line in enumerate(lines):
    if 'def _edges(self, mask):' in line:
        indent = line[:len(line) - len(line.lstrip())]
        new_lines.append(line)
        new_lines.append(indent + '    m  = mask.unsqueeze(1).float()')
        new_lines.append(indent + '    kx = self.kx.to(device=m.device, dtype=m.dtype)')
        new_lines.append(indent + '    ky = self.ky.to(device=m.device, dtype=m.dtype)')
        new_lines.append(indent + '    g  = (F.conv2d(m, kx, padding=1)**2 +')
        new_lines.append(indent + '          F.conv2d(m, ky, padding=1)**2).sqrt()')
        new_lines.append(indent + '    return g / (g.flatten(1).max(1)[0].view(-1,1,1,1) + 1e-8)')
        skip = 5  # skip next 5 lines (old body)
        continue
    if skip > 0:
        skip -= 1
        continue
    new_lines.append(line)

with open('/content/losses.py', 'w') as f:
    f.write('\n'.join(new_lines))

# Reload and patch live class
import losses as losses_mod
import torch.nn.functional as F

def _edges_fixed(self, mask):
    m  = mask.unsqueeze(1).float()
    kx = self.kx.to(device=m.device, dtype=m.dtype)
    ky = self.ky.to(device=m.device, dtype=m.dtype)
    g  = (F.conv2d(m, kx, padding=1)**2 +
          F.conv2d(m, ky, padding=1)**2).sqrt()
    return g / (g.flatten(1).max(1)[0].view(-1,1,1,1) + 1e-8)

losses_mod.BoundaryLoss._edges = _edges_fixed
print(" File rewritten + live class patched")
print(" Now run your training cell")

 File rewritten + live class patched
 Now run your training cell


In [ ]:

"""import sys, importlib


import losses
bl = losses.BoundaryLoss
import torch, torch.nn.functional as F

def _edges_fixed(self, mask):
    m  = mask.unsqueeze(1).float()
    kx = self.kx.to(m.dtype)
    ky = self.ky.to(m.dtype)
    g  = (F.conv2d(m, kx, padding=1)**2 +
          F.conv2d(m, ky, padding=1)**2).sqrt()
    return g / (g.flatten(1).max(1)[0].view(-1,1,1,1) + 1e-8)

bl._edges = _edges_fixed
print("BoundaryLoss._edges patched in memory")
"""

BoundaryLoss._edges patched in memory


In [11]:
!python train.py --data_root /content/VOCdevkit/VOC2012 --epochs 80 --batch_size 16 --lr 3e-4 --save_path best_model.pth


  EfficientSeg v3  —  targeting DSC/GFLOPs > 5
  Device=cuda  epochs=80  bs=16
  Effective batch = 32
  KD starts epoch 45
  SWA starts epoch 80
/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
Train=1464  Val=1449
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()

GFLOPs: 0.0871

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will c

In [13]:
!python validate.py --data_root /content/VOCdevkit/VOC2012 --checkpoint best_model.pth --split val --tta

Device=cuda  checkpoint=best_model.pth  split=val  tta=True
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Model loaded. Dataset size: 1449
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if n

In [15]:
!python inference.py --in_dir=/path/test_images/ --out_dir=/path/group_output/

  EfficientSeg Inference
  Device     : cuda
  Checkpoint : best_model.pth
  Input dir  : /path/test_images/
  Output dir : /path/group_output/
  TTA        : False
Model loaded 

Traceback (most recent call last):
  File "/content/inference.py", line 138, in <module>
    main()
  File "/content/inference.py", line 102, in main
    raise NotADirectoryError(f"Input directory not found: {args.in_dir}")
NotADirectoryError: Input directory not found: /path/test_images/
